# 03 — Classical innovation benchmarks

Run the Gaussian, Student-t, and joint bootstrap benchmarks through one shared experiment function.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from innovcal.experiments import run_financial_experiment

In [2]:
returns = pd.read_csv(ROOT / 'data/processed/financial_returns.csv', index_col=0).to_numpy()
result = run_financial_experiment(
    returns,
    methods=('gaussian', 'student_t', 'bootstrap'),
    lags=1, n_paths=250, seed=123,
)
result.evaluation.sort_values('energy_score')

/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: divide by zero encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: overflow encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/vector_ar/fit.py:63: RuntimeWarning: invalid value encountered in matmul
  fitted = X @ beta
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: divide by zero encountered in matmul
  shocks = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: overflow encountered in matmul
  shocks = rng.multivariate_normal(
/Users/jonathanma/Desktop/Projects/innovative-droBVAR/src/innovcal/innovations/gaussian.py:34: RuntimeWarning: invalid value encountered in matmul
  shocks = rng.

,dgp,forecast_model,innovation_model,avg_coverage,avg_width,energy_score,crps,interval_score,ece,pit_deviation,...,width_1,coverage_2,width_2,coverage_3,width_3,coverage_4,width_4,nominal_coverage,coverage_error,abs_coverage_error
0,financial,VAR,gaussian,0.894628,0.037039,0.014032,0.006034,0.046202,0.014738,0.011157,...,0.025745,0.851240,0.033599,0.867769,0.035197,0.966942,0.053614,0.9,-0.005372,0.005372
2,financial,VAR,bootstrap,0.888430,0.036546,0.014072,0.006049,0.046443,0.023967,0.011405,...,0.024257,0.867769,0.033843,0.859504,0.036013,0.975207,0.052071,0.9,-0.011570,0.011570
1,financial,VAR,student_t,0.869835,0.035086,0.014126,0.006068,0.046231,0.043251,0.010496,...,0.024370,0.859504,0.032364,0.809917,0.033313,0.966942,0.050297,0.9,-0.030165,0.030165


In [3]:
CACHE = ROOT / 'results/notebook_cache'
CACHE.mkdir(parents=True, exist_ok=True)
result.evaluation.to_csv(CACHE / 'classical_evaluation.csv', index=False)
np.savez(CACHE / 'financial_split.npz', train=result.split.train, calibration=result.split.calibration, test=result.split.test)
np.savez(CACHE / 'fitted_var.npz', beta=result.fitted_var['beta'], lags=result.fitted_var['lags'])
for name, forecast in result.forecasts.items():
    np.savez(CACHE / f'forecast_{name}.npz', **forecast)